### Configuración GPU e Importaciones

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sysconfig import get_paths
import ctypes
import os
import gc

site_packages = Path(get_paths()['purelib'])
cuda_lib_dirs = [
    site_packages / 'nvidia' / 'cuda_runtime' / 'lib',
    site_packages / 'nvidia' / 'cuda_nvrtc' / 'lib',
    site_packages / 'nvidia' / 'cublas' / 'lib',
    site_packages / 'nvidia' / 'cufft' / 'lib',
    site_packages / 'nvidia' / 'curand' / 'lib',
    site_packages / 'nvidia' / 'cusolver' / 'lib',
    site_packages / 'nvidia' / 'cusparse' / 'lib',
    site_packages / 'nvidia' / 'cudnn' / 'lib',
    site_packages / 'nvidia' / 'cuda_cupti' / 'lib',
    site_packages / 'nvidia' / 'nccl' / 'lib',
    site_packages / 'nvidia' / 'nvjitlink' / 'lib',
]

loaded_cuda_libs = 0
for lib_dir in cuda_lib_dirs:
    if lib_dir.exists():
        for lib_path in sorted(lib_dir.glob('*.so*')):
            ctypes.CDLL(str(lib_path), mode=ctypes.RTLD_GLOBAL)
            loaded_cuda_libs += 1

existing_ld = os.environ.get('LD_LIBRARY_PATH', '')
cuda_ld_paths = [str(path) for path in cuda_lib_dirs if path.exists()]
if existing_ld:
    cuda_ld_paths.append(existing_ld)
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda_ld_paths)
print(f'Librerías CUDA cargadas: {loaded_cuda_libs}')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('TensorFlow:', tf.__version__)
print('GPUs detectadas:', gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ Crecimiento de memoria habilitado.")
    except RuntimeError as e:
        print(e)

### Carga y Preprocesamiento de Datos

In [ ]:
# Cargar dataset
print("Cargando dataset...")
df = pd.read_csv('../Dataset/default of credit card clients.csv', sep=',', skiprows=1)

# Renombrar variable objetivo
if 'default payment next month' in df.columns:
    df.rename(columns={'default payment next month': 'default'}, inplace=True)

# Eliminar ID
if 'ID' in df.columns:
    df.drop('ID', axis=1, inplace=True)

print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} características")

# Separar variables
X = df.drop('default', axis=1)
y = df['default']

# Dividir train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalar datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Distribución clases - Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

### Implementación de Grey Wolf Optimizer (GWO)

In [ ]:
class GreyWolfOptimizerOptimized:
    """
    GWO Optimizado para mínimo consumo de memoria RAM
    """
    def __init__(self, n_wolves=15, max_iter=20, dim=4):
        self.n_wolves = n_wolves
        self.max_iter = max_iter
        self.dim = dim
        # Historial para animaciones (AGREGAR ESTO)
        self.history_positions = []  # Posiciones de TODOS los lobos
        self.history_alpha = []      # Posición del mejor lobo (Alpha)
        self.history_fitness = []    # Fitness del Alpha
        
        # Inicializar posiciones con valores válidos
        self.positions = np.zeros((n_wolves, dim), dtype=np.float32)
        for i in range(n_wolves):
            self.positions[i, 0] = np.random.uniform(0.001, 0.01)    # learning_rate
            self.positions[i, 1] = np.random.uniform(0.1, 0.5)       # dropout_rate
            self.positions[i, 2] = np.random.uniform(16, 64)         # neurons1
            self.positions[i, 3] = np.random.uniform(8, 32)          # neurons2
        
        self.fitness = np.full(n_wolves, np.inf, dtype=np.float32)
        
        self.alpha_pos = np.zeros(dim, dtype=np.float32)
        self.alpha_score = np.inf
        self.beta_pos = np.zeros(dim, dtype=np.float32)
        self.beta_score = np.inf
        self.delta_pos = np.zeros(dim, dtype=np.float32)
        self.delta_score = np.inf
        
        self.history = []
        
    def create_mlp(self, params, input_dim):
        """Crear MLP con hiperparámetros dados"""
        lr = max(float(params[0]), 1e-6)
        dropout = np.clip(float(params[1]), 0.05, 0.7)
        n1 = max(int(params[2]), 8)
        n2 = max(int(params[3]), 4)
        
        model = Sequential([
            Dense(n1, activation='relu', input_dim=input_dim),
            BatchNormalization(),
            Dropout(dropout),
            Dense(n2, activation='relu'),
            BatchNormalization(),
            Dropout(dropout),
            Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        return model
    
    def evaluate_fitness(self, params, X_train, y_train, X_val, y_val):
        """
        Evaluar fitness con limpieza agresiva de memoria
        """
        model = None
        history = None
        fitness = np.inf
        
        try:
            # Crear modelo
            model = self.create_mlp(params, X_train.shape[1])
            
            # Entrenar con pocas épocas para evaluación rápida
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=8,  # Reducido para velocidad
                batch_size=256,
                verbose=0,
                callbacks=[
                    EarlyStopping(
                        monitor='val_auc', 
                        mode='max', 
                        patience=2, 
                        restore_best_weights=True
                    )
                ]
            )
            
            # Obtener AUC de validación
            if 'val_auc' in history.history and len(history.history['val_auc']) > 0:
                val_auc = max(history.history['val_auc'])
                
                if not np.isnan(val_auc) and not np.isinf(val_auc):
                    fitness = 1.0 - val_auc
            
        except Exception as e:
            print(f"Error en evaluación: {e}")
            fitness = np.inf
        
        finally:
            # 🔥 LIMPIEZA AGRESIVA DE MEMORIA
            if model is not None:
                del model
            
            if history is not None:
                del history
            
            # Limpiar sesión de Keras (CRÍTICO)
            tf.keras.backend.clear_session()
            
            # Forzar garbage collection
            gc.collect()
        
        return fitness
    
    def optimize(self, X_train, y_train, X_val, y_val):
        """Ejecutar optimización guardando historial completo"""
        print(f"\n🐺 Iniciando GWO: {self.n_wolves} lobos, {self.max_iter} iteraciones")
        
        # Inicializar listas de historial
        self.history_positions = []
        self.history_alpha = []
        self.history_fitness = []
        
        # Evaluar población inicial
        print("Evaluando población inicial...")
        for i in range(self.n_wolves):
            self.fitness[i] = self.evaluate_fitness(
                self.positions[i], X_train, y_train, X_val, y_val
            )
            if i % 5 == 0:
                print(f"  Lobo {i+1}/{self.n_wolves}: fitness = {self.fitness[i]:.4f}")
        
        # Actualizar alpha, beta, delta iniciales
        sorted_indices = np.argsort(self.fitness)
        self.alpha_score = self.fitness[sorted_indices[0]]
        self.alpha_pos = self.positions[sorted_indices[0]].copy()
        self.beta_score = self.fitness[sorted_indices[1]]
        self.beta_pos = self.positions[sorted_indices[1]].copy()
        self.delta_score = self.fitness[sorted_indices[2]]
        self.delta_pos = self.positions[sorted_indices[2]].copy()
        
        # ⭐ GUARDAR ESTADO INICIAL
        self.history_positions.append(self.positions.copy())
        self.history_alpha.append(self.alpha_pos.copy())
        self.history_fitness.append(self.alpha_score)
        
        print(f"\n✅ Mejor fitness inicial: {self.alpha_score:.4f} (AUC: {1-self.alpha_score:.4f})")
        
        # Loop principal
        for iteration in range(self.max_iter):
            a = 2 * (1 - (iteration / self.max_iter) ** 2)
            
            # Actualizar posiciones
            for i in range(self.n_wolves):
                for j in range(self.dim):
                    r1, r2 = np.random.random(), np.random.random()
                    A1 = 2 * a * r1 - a
                    C1 = 2 * r2
                    D_alpha = abs(C1 * self.alpha_pos[j] - self.positions[i, j])
                    X1 = self.alpha_pos[j] - A1 * D_alpha
                    
                    r1, r2 = np.random.random(), np.random.random()
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2
                    D_beta = abs(C2 * self.beta_pos[j] - self.positions[i, j])
                    X2 = self.beta_pos[j] - A2 * D_beta
                    
                    r1, r2 = np.random.random(), np.random.random()
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2
                    D_delta = abs(C3 * self.delta_pos[j] - self.positions[i, j])
                    X3 = self.delta_pos[j] - A3 * D_delta
                    
                    self.positions[i, j] = (X1 + X2 + X3) / 3
            
            # Mantener límites
            self.positions[:, 0] = np.clip(self.positions[:, 0], 0.0001, 0.01)
            self.positions[:, 1] = np.clip(self.positions[:, 1], 0.05, 0.7)
            self.positions[:, 2] = np.clip(self.positions[:, 2], 8, 128)
            self.positions[:, 3] = np.clip(self.positions[:, 3], 4, 64)
            
            # Evaluar nueva población
            for i in range(self.n_wolves):
                self.fitness[i] = self.evaluate_fitness(
                    self.positions[i], X_train, y_train, X_val, y_val
                )
            
            # Actualizar alpha, beta, delta
            for i in range(self.n_wolves):
                if self.fitness[i] < self.alpha_score:
                    self.delta_score = self.beta_score
                    self.delta_pos = self.beta_pos.copy()
                    self.beta_score = self.alpha_score
                    self.beta_pos = self.alpha_pos.copy()
                    self.alpha_score = self.fitness[i]
                    self.alpha_pos = self.positions[i].copy()
                elif self.fitness[i] < self.beta_score:
                    self.delta_score = self.beta_score
                    self.delta_pos = self.beta_pos.copy()
                    self.beta_score = self.fitness[i]
                    self.beta_pos = self.positions[i].copy()
                elif self.fitness[i] < self.delta_score:
                    self.delta_score = self.fitness[i]
                    self.delta_pos = self.positions[i].copy()
            
            # ⭐ GUARDAR HISTORIAL EN CADA ITERACIÓN
            self.history_positions.append(self.positions.copy())
            self.history_alpha.append(self.alpha_pos.copy())
            self.history_fitness.append(self.alpha_score)
            
            if iteration % 5 == 0:
                print(f"Iter {iteration:3d} | Alpha AUC: {1-self.alpha_score:.4f} | "
                      f"LR: {self.alpha_pos[0]:.5f}, DO: {self.alpha_pos[1]:.3f}, "
                      f"N1: {int(self.alpha_pos[2])}, N2: {int(self.alpha_pos[3])}")
            
            gc.collect()
        
        # Verificar resultado final
        if np.isinf(self.alpha_score):
            print("\n⚠️ ADVERTENCIA: No se encontró solución válida.")
            self.alpha_pos = np.array([0.001, 0.2, 32, 16], dtype=np.float32)
            self.alpha_score = 0.3
        
        print(f"\n✅ Optimización completada!")
        print(f"Mejor AUC: {1-self.alpha_score:.4f}")
        print(f"Mejor configuración: LR={self.alpha_pos[0]:.5f}, Dropout={self.alpha_pos[1]:.3f}, "
              f"N1={int(self.alpha_pos[2])}, N2={int(self.alpha_pos[3])}\n")
        
        return self.alpha_pos, self.alpha_score

#### Funcion para crear las animaciones 3d

In [ ]:
def create_gwo_animation_3d(gwo, save_path='gwo_animacion_3d'):
    """
    Crea animación 3D del proceso de optimización GWO
    """
    
    # Verificar que tenemos historial
    if not hasattr(gwo, 'history_positions') or len(gwo.history_positions) == 0:
        print(" No hay historial de posiciones. Asegúrate de que el GWO guardó el historial.")
        return None
    
    # Convertir historial a arrays numpy
    history_positions = np.array(gwo.history_positions)  # (iteraciones, lobos, dim)
    history_alpha = np.array(gwo.history_alpha)          # (iteraciones, dim)
    history_fitness = np.array(gwo.history_fitness)      # (iteraciones,)
    
    n_iterations = len(history_fitness)
    n_wolves = gwo.n_wolves
    
    print(f"Creando animación 3D con {n_iterations} iteraciones y {n_wolves} lobos...")
    
    # Usar los primeros 3 parámetros: LR, Dropout, N1
    positions_3d = history_positions[:, :, :3]  # (iter, lobos, 3)
    alpha_3d = history_alpha[:, :3]              # (iter, 3)
    
    fig_3d = plt.figure(figsize=(14, 10))
    ax_3d = fig_3d.add_subplot(111, projection='3d')
    
    def update_3d(frame):
        ax_3d.clear()
        
        # Plotear lobos
        wolf_pos = positions_3d[frame]
        ax_3d.scatter(wolf_pos[:, 0], wolf_pos[:, 1], wolf_pos[:, 2], 
                     c='blue', alpha=0.6, s=100, label='Lobos', edgecolors='black', linewidth=0.5)
        
        # Plotear alpha
        alpha_pos = alpha_3d[frame]
        ax_3d.scatter([alpha_pos[0]], [alpha_pos[1]], [alpha_pos[2]], 
                     c='red', s=300, marker='*', label='Alpha', zorder=5, edgecolors='black', linewidth=1)
        
        # Líneas de conexión
        for i in range(n_wolves):
            ax_3d.plot([wolf_pos[i, 0], alpha_pos[0]],
                      [wolf_pos[i, 1], alpha_pos[1]],
                      [wolf_pos[i, 2], alpha_pos[2]], 
                      'g--', alpha=0.3, linewidth=1)
        
        # Información
        fitness = history_fitness[frame]
        auc = 1 - fitness
        ax_3d.text2D(0.02, 0.98, f'Iteración: {frame}/{n_iterations-1}\n'
                    f'Fitness: {fitness:.4f}\n'
                    f'AUC: {auc:.4f}', 
                    transform=ax_3d.transAxes, fontsize=11,
                    verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax_3d.set_xlabel('Learning Rate', fontsize=11)
        ax_3d.set_ylabel('Dropout', fontsize=11)
        ax_3d.set_zlabel('Neuronas Capa 1', fontsize=11)
        ax_3d.set_title(f'GWO - Iteración {frame}', fontsize=14, fontweight='bold')
        ax_3d.legend(loc='upper right', fontsize=10)
        
        # Ajustar límites de ejes
        ax_3d.set_xlim(positions_3d[:,:,0].min(), positions_3d[:,:,0].max())
        ax_3d.set_ylim(positions_3d[:,:,1].min(), positions_3d[:,:,1].max())
        ax_3d.set_zlim(positions_3d[:,:,2].min(), positions_3d[:,:,2].max())
    
    # Crear animación
    anim_3d = FuncAnimation(fig_3d, update_3d, frames=n_iterations, 
                           interval=800, repeat=True)
    
    # Guardar animación 3D
    try:
        anim_3d.save(f'{save_path}.gif', writer=PillowWriter(fps=2), dpi=100)
        print(f"✅ Animación 3D guardada en: {save_path}.gif")
    except Exception as e:
        print(f"❌ Error al guardar animación 3D: {e}")
    
    plt.close('all')
    
    return anim_3d

### Ejecutar Optimización GWO

In [ ]:
# Dividir datos para GWO
X_train_gwo, X_val, y_train_gwo, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Convertir a float32
X_train_gwo = X_train_gwo.astype(np.float32)
X_val = X_val.astype(np.float32)
y_train_gwo = y_train_gwo.astype(np.float32)
y_val = y_val.astype(np.float32)

print("🐺 Iniciando optimización GWO con guardado de historial...")

# Inicializar GWO
gwo = GreyWolfOptimizerOptimized(
    n_wolves=15,
    max_iter=20,
    dim=4
)

# Ejecutar optimización (esto guardará el historial completo)
best_params, best_fitness = gwo.optimize(
    X_train_gwo, y_train_gwo, X_val, y_val
)

# Extraer parámetros
best_lr = float(best_params[0])
best_dropout = float(best_params[1])
best_n1 = int(best_params[2])
best_n2 = int(best_params[3])

print(f"\n✅ GWO completado!")
print(f"Mejores parámetros: LR={best_lr}, Dropout={best_dropout}, N1={best_n1}, N2={best_n2}")


#### Generar animaciones

In [ ]:
anim_3d = create_gwo_animation_3d(gwo, save_path='gwo_animacion_3d')
display(Image(filename='gwo_animacion_3d.gif'))

### Entrenar Modelo Final con Mejores Parámetros

In [ ]:
# Crear modelo final con mejores hiperparámetros
print("\n🚀 Entrenando modelo final con hiperparámetros optimizados...")

final_model = Sequential([
    Dense(best_n1, activation='relu', input_dim=X_train_scaled.shape[1]),
    BatchNormalization(),
    Dropout(best_dropout),
    Dense(best_n2, activation='relu'),
    BatchNormalization(),
    Dropout(best_dropout),
    Dense(1, activation='sigmoid')
])

final_model.compile(
    optimizer=Adam(learning_rate=best_lr),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Calcular class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {0: class_weights[0], 1: class_weights[1]}

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

# Entrenar
history = final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=500,
    batch_size=256,
    class_weight=class_weights_dict,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\n✅ Modelo final entrenado exitosamente!")

### Evaluación y Métricas

In [ ]:
# Evaluar en test set
print("\n📊 EVALUACIÓN DEL MODELO HÍBRIDO GWO-MLP")
print("="*60)

y_pred_prob = final_model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*60)

# Matriz de confusión
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cumple (0)', 'Incumple (1)'],
            yticklabels=['Cumple (0)', 'Incumple (1)'])
plt.title('Matriz de Confusión - Modelo Híbrido GWO-MLP')
plt.ylabel('Valor Real')
plt.xlabel('Predicción')
plt.show()

print(f"\nTP: {cm[1,1]} | FP: {cm[0,1]}")
print(f"FN: {cm[1,0]} | TN: {cm[0,0]}")

### Visualización de Convergencia GWO

In [ ]:
# Visualizar convergencia de GWO
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
iterations = [h['iteration'] for h in gwo.history]
scores = [h['alpha_score'] for h in gwo.history]
plt.plot(iterations, scores, 'b-o', linewidth=2, markersize=6)
plt.xlabel('Iteración')
plt.ylabel('Val Loss (Alpha)')
plt.title('Convergencia de GWO')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['auc'], label='Train AUC')
plt.plot(history.history['val_auc'], label='Val AUC')
plt.xlabel('Epoch')
plt.ylabel('Metric')
plt.title('Historial de Entrenamiento - Modelo Final')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()